# ACE++ GRPO Training Notebook

This notebook is a minimal, GPU-ready RL training workflow for ACE++ using:
- `trl.GRPOTrainer`
- `unsloth.FastLanguageModel`
- the existing `ACEOpenEnv` wrapper

It is optimized for correctness, clarity, and observable learning rather than absolute benchmark performance.

## Cell 1 — Install Dependencies

In [ ]:
!pip -q install unsloth transformers trl accelerate bitsandbytes datasets peft matplotlib

## Cell 2 — Load Model with Unsloth

In [ ]:
import json
import os
import random
import re
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from datasets import Dataset
from transformers import set_seed
from trl import GRPOConfig, GRPOTrainer
from unsloth import FastLanguageModel

set_seed(42)
random.seed(42)

assert torch.cuda.is_available(), "This notebook is intended for a GPU runtime."

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 512
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.config.use_cache = False
print("Loaded model:", MODEL_NAME)

## Cell 3 — Import Environment and Build Env Reward Wrapper

In [ ]:
from ace_env_openenv import ACEOpenEnv

ROUND_TYPES = ["cooperative", "competitive", "resource"]


def extract_first_json_object(text: str):
    start = text.find("{")
    if start == -1:
        return None
    decoder = json.JSONDecoder()
    for idx in range(start, len(text)):
        if text[idx] != "{":
            continue
        try:
            obj, _ = decoder.raw_decode(text[idx:])
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict):
            return obj
    return None


def safe_parse_completion(text: str):
    obj = extract_first_json_object(text)
    valid = True

    if not isinstance(obj, dict):
        valid = False
        obj = {}

    belief = obj.get("belief") if isinstance(obj.get("belief"), dict) else {}
    predicted_round = obj.get("predicted_round") or belief.get("predicted_round")
    if predicted_round not in ROUND_TYPES:
        valid = False
        predicted_round = "resource"

    action_field = obj.get("action")
    parameters = obj.get("parameters") if isinstance(obj.get("parameters"), dict) else {}

    if isinstance(action_field, dict):
        action = action_field.get("tool") or action_field.get("name")
        parameters = action_field.get("parameters") if isinstance(action_field.get("parameters"), dict) else parameters
    else:
        action = action_field

    if action not in {"submit_bid", "allocate_resources", "execute_contract"}:
        valid = False
        action = "allocate_resources" if predicted_round == "resource" else "submit_bid"

    try:
        amount = float(parameters.get("amount", 50))
    except Exception:
        valid = False
        amount = 50.0

    return {
        "predicted_round": predicted_round,
        "action": action,
        "parameters": {"amount": amount},
    }, valid


def build_single_round_env(ground_truth: str, payoff_seed: int, difficulty: str = "easy"):
    env = ACEOpenEnv(
        num_rounds=1,
        seed=0,
        round_type_schedule=[ground_truth],
        difficulty=difficulty,
    )
    env.reset()
    inner = getattr(env, "_env", env)
    inner.current_round_type = ground_truth
    inner.current_payoff_seed = int(payoff_seed)
    inner.current_payoff = inner._payoff_from_seed(int(payoff_seed))
    return env


def run_env_step(prompt_output: str, ground_truth: str, payoff_seed: int, difficulty: str = "easy"):
    env = build_single_round_env(ground_truth, payoff_seed, difficulty=difficulty)
    parsed, valid = safe_parse_completion(prompt_output)

    action_json = json.dumps(
        {
            "belief": {
                "predicted_round": parsed["predicted_round"],
                "confidence": 0.8 if valid else 0.2,
            },
            "action": {
                "tool": parsed["action"],
                "parameters": parsed["parameters"],
            },
        }
    )

    _, reward, _, info = env.step(action_json)

    # Explicit format signal so the model gets a dense reward even early on.
    reward = float(reward) + (0.25 if valid else -0.50)

    return {
        "reward": reward,
        "correct": bool(info.get("correct_inference", False)),
        "valid": valid,
        "info": info,
        "parsed": parsed,
    }

## Cell 4 — Define Prompt Format

In [ ]:
SYSTEM_INSTRUCTION = """You are an economic agent in a partially observable market.
Your task is to infer the hidden round type from the market_state and choose a valid tool action.

Hidden round types:
- cooperative
- competitive
- resource

Output ONLY valid JSON in exactly this format:
{
  \"predicted_round\": \"cooperative|competitive|resource\",
  \"action\": \"submit_bid|allocate_resources|execute_contract\",
  \"parameters\": {\"amount\": 50}
}
"""


def format_prompt(observation: dict):
    history = observation.get("history", [])
    history_text = json.dumps(history, indent=2)
    market_text = json.dumps(observation["market_state"], indent=2)
    return f"""{SYSTEM_INSTRUCTION}

History:
{history_text}

Current market_state:
{market_text}

Return ONLY the JSON object."""


def build_prompt_dataset(n_samples: int = 64, difficulty: str = "easy", seed: int = 123):
    rows = []
    for i in range(n_samples):
        env = ACEOpenEnv(num_rounds=1, seed=seed + i, difficulty=difficulty)
        observation = env.reset()
        state = env.state()
        rows.append(
            {
                "prompt": format_prompt(observation),
                "ground_truth": state["current_round_type"],
                "payoff_seed": int(state["current_payoff_seed"]),
                "difficulty": difficulty,
            }
        )
    return Dataset.from_list(rows)


train_dataset = build_prompt_dataset(n_samples=200, difficulty="easy", seed=123)
eval_dataset = build_prompt_dataset(n_samples=24, difficulty="easy", seed=999)
train_dataset[0]

## Cell 5 — Rollout and Reward Functions

In [ ]:
reward_log = []


@torch.no_grad()
def generate_completions(prompts, model, tokenizer, max_new_tokens: int = 96, temperature: float = 0.0):
    model.eval()
    tokenized = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if temperature > 0:
        generation_kwargs.update({"do_sample": True, "temperature": temperature, "top_p": 0.95})
    else:
        generation_kwargs.update({"do_sample": False})

    outputs = model.generate(**tokenized, **generation_kwargs)

    completions = []
    input_lengths = tokenized["attention_mask"].sum(dim=1).tolist()
    for row_idx, input_len in enumerate(input_lengths):
        completion_ids = outputs[row_idx, int(input_len):]
        completions.append(tokenizer.decode(completion_ids, skip_special_tokens=True).strip())
    return completions


def generate_and_score(prompts, rows, model=model, tokenizer=tokenizer, max_new_tokens: int = 96, temperature: float = 0.0):
    completions = generate_completions(prompts, model=model, tokenizer=tokenizer, max_new_tokens=max_new_tokens, temperature=temperature)
    rewards = []
    results = []
    for completion, row in zip(completions, rows):
        result = run_env_step(completion, row["ground_truth"], row["payoff_seed"], row["difficulty"])
        rewards.append(result["reward"])
        results.append(result)
    return completions, rewards, results


def reward_func(prompts, completions, ground_truth, payoff_seed, difficulty, trainer_state=None, **kwargs):
    rewards = []
    correct = 0
    valid = 0

    for completion, gt, ps, diff in zip(completions, ground_truth, payoff_seed, difficulty):
        result = run_env_step(completion, gt, ps, diff)
        rewards.append(result["reward"])
        correct += int(result["correct"])
        valid += int(result["valid"])

    reward_log.append(
        {
            "step": getattr(trainer_state, "global_step", len(reward_log)),
            "avg_reward": sum(rewards) / max(1, len(rewards)),
            "accuracy": correct / max(1, len(rewards)),
            "invalid_rate": 1.0 - (valid / max(1, len(rewards))),
        }
    )
    return rewards


def evaluate_model(model, tokenizer, dataset, n_rollouts: int = 8):
    rows = [dataset[i] for i in range(min(n_rollouts, len(dataset)))]
    prompts = [row["prompt"] for row in rows]
    completions, rewards, results = generate_and_score(prompts, rows, model=model, tokenizer=tokenizer, max_new_tokens=96, temperature=0.0)

    metrics = {
        "mean_reward": sum(rewards) / max(1, len(rewards)),
        "accuracy": sum(int(r["correct"]) for r in results) / max(1, len(results)),
        "valid_rate": sum(int(r["valid"]) for r in results) / max(1, len(results)),
        "samples": [
            {
                "completion": completion,
                "reward": reward,
                "parsed": result["parsed"],
                "correct": result["correct"],
                "valid": result["valid"],
            }
            for completion, reward, result in zip(completions[:3], rewards[:3], results[:3])
        ],
    }
    return metrics

## Cell 6 — GRPO Configuration

In [ ]:
bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = GRPOConfig(
    output_dir="ace_grpo_outputs",
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    num_generations=2,
    max_steps=120,
    logging_steps=1,
    save_steps=60,
    save_strategy="steps",
    report_to="none",
    remove_unused_columns=False,
    bf16=bf16_supported,
    fp16=not bf16_supported,
    gradient_checkpointing=True,
    max_completion_length=96,
    temperature=0.9,
    top_p=0.95,
)

training_args

## Cell 7 — Trainer Setup

In [ ]:
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=[reward_func],
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("Trainer ready.")

## Cell 8 — Training Loop

In [ ]:
baseline_metrics = evaluate_model(model, tokenizer, eval_dataset, n_rollouts=8)
print("Before training:")
print(json.dumps({k: v for k, v in baseline_metrics.items() if k != 'samples'}, indent=2))
print("Sample completions before training:")
for sample in baseline_metrics["samples"]:
    print("-" * 80)
    print(sample["completion"])
    print(sample["parsed"], "reward=", round(sample["reward"], 3), "correct=", sample["correct"], "valid=", sample["valid"])

train_result = trainer.train()
print(train_result)
print("Logged reward entries:", len(reward_log))

## Cell 9 — Evaluation

In [ ]:
after_metrics = evaluate_model(model, tokenizer, eval_dataset, n_rollouts=8)

print("After training:")
print(json.dumps({k: v for k, v in after_metrics.items() if k != 'samples'}, indent=2))

print("\nImprovement summary:")
print(f"Reward:   {baseline_metrics['mean_reward']:.3f} -> {after_metrics['mean_reward']:.3f}")
print(f"Accuracy: {baseline_metrics['accuracy']:.3f} -> {after_metrics['accuracy']:.3f}")
print(f"Valid:    {baseline_metrics['valid_rate']:.3f} -> {after_metrics['valid_rate']:.3f}")

print("\nSample completions after training:")
for sample in after_metrics["samples"]:
    print("-" * 80)
    print(sample["completion"])
    print(sample["parsed"], "reward=", round(sample["reward"], 3), "correct=", sample["correct"], "valid=", sample["valid"])

## Cell 10 — Save Model

In [ ]:
SAVE_DIR = "ace_qwen_grpo_adapter"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save the LoRA adapter and tokenizer only. Do not merge a 4-bit model here.
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved adapter to", SAVE_DIR)

## Cell 11 — Visualization

In [ ]:
reward_steps = list(range(len(reward_log)))
avg_rewards = [row["avg_reward"] for row in reward_log]
accuracies = [row["accuracy"] for row in reward_log]
invalid_rates = [row["invalid_rate"] for row in reward_log]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: reward + accuracy over training
axes[0].plot(reward_steps, avg_rewards, label="Avg reward", color="steelblue")
axes[0].plot(reward_steps, accuracies, label="Accuracy", color="seagreen", linestyle="--")
axes[0].axhline(y=0.333, color="gray", linestyle=":", alpha=0.6, label="Random baseline (33%)")
axes[0].set_title("Reward & Accuracy — GRPO Training")
axes[0].set_xlabel("Logged step")
axes[0].set_ylabel("Value")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: before/after eval bar chart on a separate axis
labels = ["Before training", "After training"]
acc_vals = [baseline_metrics["accuracy"], after_metrics["accuracy"]]
rew_vals = [baseline_metrics["mean_reward"], after_metrics["mean_reward"]]

x = [0, 1]
axes[1].bar(x, acc_vals, width=0.35, alpha=0.75, label="Inference accuracy", color="seagreen")
axes[1].bar([xi + 0.38 for xi in x], rew_vals, width=0.35, alpha=0.75, label="Mean reward", color="steelblue")
axes[1].set_xticks([0.19, 1.19], labels)
axes[1].set_title("Before vs After — Eval Set")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
print("Saved: training_curves.png")
plt.show()

## Notes

- Training is intentionally small so it finishes on a single Hugging Face / Colab GPU.
- The reward uses the real ACE environment step, plus a small JSON-format bonus/penalty for dense learning signal.
- If improvement is noisy, increase `max_steps` modestly or keep the dataset on `difficulty=\"easy\"` for clearer signal.